# 01 · Pre-processing
Clean the crime and street-light data, count them in 1 km squares and pick out the urban study area.

In [ ]:
from pathlib import Path

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

## 1. Crime data

In [ ]:
crime_files = sorted(Path('../data/raw/police').rglob('*-street.csv'))
crime = pd.concat([pd.read_csv(f) for f in crime_files])
crime.shape

In [ ]:
crime['Crime type'].value_counts()

Remove anti-social behaviour (incidents, not crimes).

In [ ]:
crime = crime[crime['Crime type'] != 'Anti-social behaviour']
crime.isna().sum()

Drop rows without coordinates. PSNI reuses Crime IDs across months, so check duplicates on ID **and** month.

In [ ]:
crime = crime.dropna(subset=['Longitude', 'Latitude'])
crime = crime.drop_duplicates(subset=['Crime ID', 'Month'])
crime.shape

Split into **street crime** (outdoor, lighting-sensitive) and a **comparison** group (mostly indoor).

In [ ]:
street_types = ['Robbery', 'Theft from the person', 'Vehicle crime',
                'Criminal damage and arson']
comparison_types = ['Violence and sexual offences', 'Shoplifting']

crime['group'] = 'other'
crime.loc[crime['Crime type'].isin(street_types), 'group'] = 'street'
crime.loc[crime['Crime type'].isin(comparison_types), 'group'] = 'comparison'
crime['group'].value_counts()

Make points and convert to the Irish Grid (EPSG:29902).

In [ ]:
crime_map = gpd.GeoDataFrame(
    crime,
    geometry=gpd.points_from_xy(crime.Longitude, crime.Latitude),
    crs='EPSG:4326'
)
crime_map = crime_map.to_crs(epsg=29902)

## 2. Street lights
Keep coordinates inside Northern Ireland and count each column once.

In [ ]:
lights = pd.read_csv('../data/raw/streetlights/Street Lighting Assets.csv', low_memory=False)
lights.shape

In [ ]:
lights = lights[lights['EASTING'].between(180000, 370000) &
                lights['NORTHING'].between(300000, 460000)]
lights = lights.drop_duplicates(subset=['EASTING', 'NORTHING'])
lights.shape

In [ ]:
lights['LED'] = lights['LANTERN_BOQ_NAME'].str.upper().str.contains('LED', na=False)
lights['LED'].mean()

In [ ]:
lights_map = gpd.GeoDataFrame(
    lights,
    geometry=gpd.points_from_xy(lights.EASTING, lights.NORTHING),
    crs='EPSG:29902'
)

## 3. Grid and population

In [ ]:
grid = gpd.read_file('../data/raw/nisra_grid/Grid_Square_2021_1km.shp')
grid = grid.to_crs(epsg=29902)
grid = grid.rename(columns={'GSQ21_1km': 'cell_id'})[['cell_id', 'geometry']]

census = pd.read_csv('../data/raw/nisra_grid/census-2021-grid-square-product-1km-full-dataset.csv',
                     usecols=['Geography', 'PERSONS'])
census = census.rename(columns={'Geography': 'cell_id', 'PERSONS': 'population'})

grid = grid.merge(census, on='cell_id', how='left')
grid['population'] = grid['population'].fillna(0)
grid.shape

## 4. Counts per square

In [ ]:
crime_in_grid = gpd.sjoin(crime_map, grid[['cell_id', 'geometry']], predicate='intersects')
crime_in_grid = crime_in_grid[~crime_in_grid.index.duplicated()]

lights_in_grid = gpd.sjoin(lights_map, grid[['cell_id', 'geometry']], predicate='intersects')
lights_in_grid = lights_in_grid[~lights_in_grid.index.duplicated()]

print('Crimes placed:', len(crime_in_grid))
print('Lights placed:', len(lights_in_grid))

In [ ]:
for group in ['street', 'comparison']:
    counts = crime_in_grid[crime_in_grid['group'] == group].groupby('cell_id').size()
    grid[group + '_count'] = grid['cell_id'].map(counts).fillna(0)

grid['n_lights'] = grid['cell_id'].map(lights_in_grid.groupby('cell_id').size()).fillna(0)
grid['pct_led'] = grid['cell_id'].map(lights_in_grid.groupby('cell_id')['LED'].mean() * 100)

Optional: shops, pubs and takeaways from OpenStreetMap.

In [ ]:
poi_file = Path('../data/raw/osm/gis_osm_pois_free_1.shp')
if poi_file.exists():
    pois = gpd.read_file(poi_file).to_crs(epsg=29902)
    commercial = ['pub', 'bar', 'nightclub', 'restaurant', 'fast_food', 'cafe',
                  'supermarket', 'convenience', 'department_store', 'mall', 'clothes']
    pois = pois[pois['fclass'].isin(commercial)]
    pois_in_grid = gpd.sjoin(pois, grid[['cell_id', 'geometry']], predicate='intersects')
    pois_in_grid = pois_in_grid[~pois_in_grid.index.duplicated()]
    grid['n_poi'] = grid['cell_id'].map(pois_in_grid.groupby('cell_id').size()).fillna(0)
else:
    print('No OpenStreetMap file found - skipping commercial activity.')

## 5. Urban study area
Share of each square inside a Band A–E settlement (5,000+ people).

In [ ]:
settlements = gpd.read_file('../data/raw/settlements/settlements-2015-above-500-threshold.shp')
settlements = settlements.to_crs(epsg=29902)
urban_settlements = settlements[settlements['Band'].isin(['A', 'B', 'C', 'D', 'E'])]

pieces = gpd.overlay(grid[['cell_id', 'geometry']],
                     urban_settlements[['Name', 'Band', 'geometry']], how='intersection')
pieces['share'] = pieces.area / 1_000_000

grid['urban_share'] = grid['cell_id'].map(pieces.groupby('cell_id')['share'].sum()).fillna(0)
largest = pieces.sort_values('share').drop_duplicates('cell_id', keep='last').set_index('cell_id')
grid['settlement'] = grid['cell_id'].map(largest['Name'])

Keep squares at least half urban with 100+ residents.

In [ ]:
grid['study_area'] = (grid['urban_share'] >= 0.5) & (grid['population'] >= 100)
study = grid[grid['study_area']].copy()
print('Study squares:', len(study))
print('Settlements:', study['settlement'].nunique())

In [ ]:
ax = grid.plot(color='lightgrey', figsize=(7, 7))
study.plot(ax=ax, color='tab:blue')
plt.title('Study area: urban 1 km squares')
plt.axis('off')
plt.show()

## 6. Rates and lighting index
Rates per 1,000 residents.

In [ ]:
for group in ['street', 'comparison']:
    study[group + '_rate'] = study[group + '_count'] / study['population'] * 1000

study['lights_per_km2'] = study['n_lights']          # every square is 1 km²
study['lights_per_1000'] = study['n_lights'] / study['population'] * 1000
study[['street_rate', 'comparison_rate', 'lights_per_km2', 'lights_per_1000']].describe()

Street Light Availability Index (SLAI): average percentile of lights per km² and lights per 1,000 residents.

In [ ]:
study['slai'] = (study['lights_per_km2'].rank(pct=True) +
                 study['lights_per_1000'].rank(pct=True)) / 2
study['light_class'] = pd.qcut(study['slai'], 3, labels=['Low', 'Medium', 'High'])
study['light_class'].value_counts()

Label each square with its 10 km block (used for spatial cross-validation), then save.

In [ ]:
study['x'] = study.geometry.centroid.x
study['y'] = study.geometry.centroid.y
study['block'] = (study['x'] // 10000).astype(int).astype(str) + '_' + \
                 (study['y'] // 10000).astype(int).astype(str)
study['block'].nunique()

In [ ]:
Path('../data/processed').mkdir(parents=True, exist_ok=True)
study['light_class'] = study['light_class'].astype(str)
study.to_file('../data/processed/study_area.gpkg')